In [ ]:
# --- Configuración del entorno (portable: local / Colab) ---
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Project-ComputerVision')
    IS_COLAB = True
except ImportError:
    PROJECT_ROOT = Path().resolve().parent.parent
    IS_COLAB = False

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import CFG  # noqa: E402

print(f"Entorno: {'Colab' if IS_COLAB else 'Local'}")
print(f"Raíz del proyecto: {PROJECT_ROOT}")

# 04 · Evaluación del mejor modelo e interpretabilidad

Evaluación del modelo ganador (ResNet50 + TabNet) sobre el set de prueba: reporte de clasificación, matriz de confusión, análisis de error por escala Fitzpatrick, Grad-CAM y máscaras de atención de TabNet.

In [ ]:
# Libraries

import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import os
from pathlib import Path

In [ ]:
%pip install -q tabnet-keras

In [ ]:
# Import TabNetClassifier and TabNetEncoder
from tabnet_keras import TabNetClassifier
from tabnet_keras.encoder import TabNetEncoder

In [ ]:
custom_objects = {
    'TabNetClassifier': TabNetClassifier,
    'TabNetEncoder': TabNetEncoder
}

## Mejor modelo a nivel Básico

En este se observó

In [ ]:
ruta_archivo_modelo = str(PROJECT_ROOT + '/models/modelResNet50TabNet_pro.keras')

def cargar_modelo_keras(ruta_archivo):
    """
    Carga un modelo Keras (.keras) desde la ruta especificada.

    Args:
        ruta_archivo (str or pathlib.Path): La ruta completa al archivo del modelo.

    Returns:
        tf.keras.Model: El modelo cargado, o None si ocurre un error.
    """
    print(f"Intentando cargar el modelo desde: {ruta_archivo}")
    try:
        # Carga el modelo
        model = tf.keras.models.load_model(ruta_archivo)

        print("\n¡Modelo cargado exitosamente!")
        model.summary()
        return model

    except FileNotFoundError:
        print(f"Error: No se encontró el archivo en la ruta especificada: {ruta_archivo}")
        return None
    except Exception as e:
        # Captura cualquier otro error (ej. archivo corrupto, capas personalizadas faltantes)
        print(f"Error al cargar el modelo: {e}")
        return None

In [ ]:
bestModel = tf.keras.models.load_model(ruta_archivo_modelo, custom_objects)

Obteniendo las mejores métricas:

In [ ]:
filename = str(PROJECT_ROOT + '/preprocessed_multimodal/preprocessed_multimodal_data_brasil.pkl')
loaded_data = pd.read_pickle(filename)

df_train = loaded_data['df_train']
df_train_balanced = loaded_data['df_train_balanced']

df_val = loaded_data['df_val']
df_test = loaded_data['df_test']

X_train_processed = loaded_data['X_train_processed']
# X_train_processed_balanced = loaded_data['X_train_balanced'],
X_val_processed = loaded_data['X_val_processed']
X_test_processed = loaded_data['X_test_processed']

y_train = loaded_data['y_train']
y_train_balanced = loaded_data['y_train_balanced']

y_val = loaded_data['y_val']
y_test = loaded_data['y_test']

preprocessor = loaded_data['preprocessor']

print("¡Datos cargados y listos para el modelado!")

In [ ]:
X_test_processed.shape

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input

In [ ]:
IMG_SHAPE = (256, 256, 3)
BATCH_SIZE = 64

In [ ]:
local_img_path = str(PROJECT_ROOT + '/Data/zr7vgbcyr2-1/PAD_UFES_20_ALL_IMAGES')

In [ ]:
val_image_paths_local = [os.path.join(local_img_path, fname) for fname in df_val['img_id']]
test_image_paths_local = [os.path.join(local_img_path, fname) for fname in df_test['img_id']]

In [ ]:
def load_and_resize_image(image_path, tabular_row, label):
    image = tf.io.read_file(image_path)
    image = tf.io.decode_image(image, channels=3)
    image.set_shape([None, None, 3])

    image = tf.image.resize(image, [IMG_SHAPE[0], IMG_SHAPE[1]])
    image = tf.cast(image, tf.float32)

    return (image, tabular_row), label

def create_fast_dataset(image_paths, tabular_data, labels, augment=False, augment_fn=None):
    """
    Crea un dataset tf.data de alto rendimiento para el modelo multimodal.

    Args:
        image_paths: Lista de rutas a las imágenes.
        tabular_data: Array de datos tabulares preprocesados.
        labels: Array de etiquetas (one-hot).
        augment: Si es True, se aplica aumentación de datos.
        augment_fn: Función de aumentación (p. ej. un pipeline Keras).
            Solo se usa si augment=True.
    """
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, tabular_data, labels))
    dataset = dataset.map(load_and_resize_image, num_parallel_calls=tf.data.AUTOTUNE)

    if augment and augment_fn is not None:
        dataset = dataset.map(lambda x, y: ((augment_fn(x[0]), x[1]), y),
                              num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.map(lambda x, y: ((preprocess_input(x[0]), x[1]), y),
                          num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        dataset = dataset.shuffle(buffer_size=1024)

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

In [ ]:
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

In [ ]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
# y_train_encoded_balanced = label_encoder.fit_transform(y_train_balanced)

y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

print("Ejemplo de etiqueta original:", y_train.iloc[0])
print("Después de LabelEncoder:", y_train_encoded[0])
print("Clases encontradas por el encoder:", label_encoder.classes_)
num_classes = len(label_encoder.classes_)

y_train_cat = to_categorical(y_train_encoded, num_classes=num_classes)
y_val_cat = to_categorical(y_val_encoded, num_classes=num_classes)
y_test_cat = to_categorical(y_test_encoded, num_classes=num_classes)

print("\nDespués de to_categorical:", y_train_cat[0])

In [ ]:
ds_val = create_fast_dataset(
    val_image_paths_local,
    X_val_processed,
    y_val_cat,
    augment=False
)

ds_test = create_fast_dataset(
    test_image_paths_local,
    X_test_processed,
    y_test_cat,
    augment=False
)

Desarrolo de la arquitecturas de los modelos:

In [ ]:
import tensorflow as tf
from tensorflow import keras

In [ ]:
try:
    model = keras.applications.Xception(weights=None)

    keras.utils.plot_model(
        model,
        # 1. Cambiamos de nuevo a .png
        to_file='xception_architecture_horizontal.png',
        show_shapes=True,
        show_layer_names=True,
        show_layer_activations=True,
        expand_nested=True,

        # 2. Mantenemos el DPI alto para buena calidad
        dpi=300,

        # 3. Mantenemos la orientación horizontal
        rankdir='LR'
    )

    print("\n¡Diagrama guardado como 'xception_architecture_horizontal.png' (horizontal y alta resolución)!")

except ImportError:
    print("\nError: Faltan 'pydot' o 'graphviz'.")
    print("Asegúrate de que Graphviz esté instalado y en tu PATH.")
except Exception as e:
    print(f"\nOcurrió un error: {e}")

***

Analizando el mejor resultado del modelo

In [ ]:
y_pred_raw = bestModel.predict(ds_test)
y_pred_classes = np.argmax(y_pred_raw, axis=1)

In [ ]:
y_true_classes = label_encoder.inverse_transform(y_test_encoded)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
y_test_classes = label_encoder.inverse_transform(y_test_encoded)

In [ ]:
np.unique(y_pred_classes)

In [ ]:
np.unique(y_test_encoded)

In [ ]:
target_names = ['ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK']

print("\n--- Reporte de Clasificación (Test Set) ---")
print(classification_report(y_test_encoded, y_pred_classes, target_names=target_names))

In [ ]:
cm = confusion_matrix(y_test_encoded, y_pred_classes, labels=[0, 1, 2, 3, 4, 5])

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='g', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicho por el Modelo')
plt.ylabel('Valor Real (Verdadero)')
plt.title('Matriz de Confusión')
plt.show()

## Análisis de Error

In [ ]:
df_test['fitspatrick']

In [ ]:
df = pd.DataFrame({
    'y_true': y_test_encoded,
    'y_pred': y_pred_classes,
    'fitzpatrick': df_test['fitspatrick']
})

df['es_error'] = df['y_true'] != df['y_pred']

error_rate_by_fitz = df.groupby('fitzpatrick')['es_error'].mean()

print("--- Tasa de Error por Escala Fitzpatrick ---")
print(error_rate_by_fitz * 100)

In [ ]:
df['img_id'] = df_test['img_id']

In [ ]:
df[(df['es_error'] == 0) & (df['fitzpatrick'] == 5)]

3: PAT_856_1626_630.png vs PAT_746_1407_708.png

4: PAT_403_804_994.png vs PAT_596_1146_294.png

5: PAT_352_729_947.png vs PAT_67_104_695.png

***

# Interpretabilidad

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import cv2
import matplotlib.pyplot as plt

def get_img_array(img_path, size):
    img = keras.utils.load_img(img_path, target_size=size)
    array = keras.utils.img_to_array(img)
    array = np.expand_dims(array, axis=0)
    return array

def make_gradcam_heatmap(img_array, tab_data, model, last_conv_layer_name, target_class_index):

    base_model = model.get_layer('resnet50')
    conv_layer = base_model.get_layer(last_conv_layer_name)

    # Use the loaded model's inputs and outputs directly
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[conv_layer.output, model.output]
    )

    with tf.GradientTape() as tape:
        # Ensure the inputs are passed in the correct order as expected by the model
        (conv_outputs, predictions) = grad_model([img_array, tab_data])
        loss = predictions[:, target_class_index]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()


def save_and_display_gradcam(img_path, heatmap, cam_path="grad_cam.jpg", alpha=0.4):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

    heatmap = np.uint8(255 * heatmap)

    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed_img = heatmap * alpha + img * (1 - alpha)
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)

    plt.imshow(superimposed_img)
    plt.axis('off')
    plt.title("Grad-CAM")
    plt.show()

    return superimposed_img

MODEL_PATH = str(PROJECT_ROOT + "/models/modelResNet50TabNet_pro.keras")
LAST_CONV_LAYER_NAME = 'conv5_block3_out'
IMG_SHAPE = (256, 256, 3)
TABULAR_SHAPE = (29,)
TARGET_CLASS_INDEX = 3

TEST_IMAGE_PATH = str(PROJECT_ROOT + '/Data/zr7vgbcyr2-1/PAD_UFES_20_ALL_IMAGES/PAT_67_104_695.png')
# Use a sample from the processed test data
TEST_TABULAR_DATA = loaded_data['X_test_processed'][0:1]


try:
    model = keras.models.load_model(MODEL_PATH, custom_objects=custom_objects) # Pass custom_objects
except Exception as e:
    print(f"Error cargando modelo: {e}")
    print("Asegúrate de pasar tus objetos personalizados si los usaste (ej. TabNetClassifier)")

img_array_for_model = keras.applications.resnet50.preprocess_input(
    get_img_array(TEST_IMAGE_PATH, size=IMG_SHAPE[:2])
)

heatmap = make_gradcam_heatmap(
    img_array_for_model,
    TEST_TABULAR_DATA,
    model,
    LAST_CONV_LAYER_NAME,
    TARGET_CLASS_INDEX
)

save_and_display_gradcam(TEST_IMAGE_PATH, heatmap)

In [ ]:
MODEL_PATH = str(PROJECT_ROOT + "/models/modelResNet50TabNet_pro.keras")
IMG_SHAPE = (256, 256, 3) # Updated from (224, 224, 3)
TABULAR_SHAPE = (29,) # Updated from (21,)
TABNET_ENCODER_NAME = 'tabnet_encoder'

# Updated FEATURE_NAMES to match the expected tabular data shape of 29
FEATURE_NAMES = [
    'smoke', 'drink', 'background_father', 'background_mother', 'age',
    'pesticide', 'gender', 'skin_cancer_history', 'cancer_history',
    'has_piped_water', 'has_sewage_system', 'fitspatrick', 'region',
    'diameter_1', 'diameter_2', 'itch', 'grew', 'hurt',
    'changed', 'bleed', 'elevation', 'feature_22', 'feature_23', 'feature_24',
    'feature_25', 'feature_26', 'feature_27', 'feature_28', 'feature_29'
]


try:
    model = keras.models.load_model(MODEL_PATH)
except Exception as e:
    print(f"Error cargando modelo: {e}")
    print("Asegúrate de pasar tus objetos personalizados (ej. TabNetClassifier) en custom_objects")

TEST_IMAGE_DATA = np.random.rand(1, *IMG_SHAPE)
# Use a sample from the processed test data and ensure shape (1, 29)
TEST_TABULAR_DATA = loaded_data['X_test_processed'][0:1]


try:
    model = keras.models.load_model(MODEL_PATH, custom_objects=custom_objects) # Pass custom_objects

    predictions = model.predict([TEST_IMAGE_DATA, TEST_TABULAR_DATA])

    tabnet_encoder_layer = model.get_layer(TABNET_ENCODER_NAME)

    masks = tabnet_encoder_layer.masks

    if masks is None:
        raise ValueError("No se encontraron máscaras. Asegúrate de que el modelo haya ejecutado una predicción.")

    step_masks = [mask.numpy() for mask in masks]
    stacked_masks = np.stack(step_masks, axis=0)

    sample_masks = stacked_masks[:, 0, :]

    feature_importances = np.sum(sample_masks, axis=0)

    feature_importances_normalized = feature_importances / np.sum(feature_importances)

    sorted_indices = np.argsort(feature_importances_normalized)
    sorted_names = [FEATURE_NAMES[i] for i in sorted_indices]
    sorted_importances = feature_importances_normalized[sorted_indices]

    plt.figure(figsize=(10, 8))
    plt.barh(sorted_names, sorted_importances)
    plt.title("Importancia de Features Tabulares (Máscaras de TabNet)")
    plt.xlabel("Importancia Normalizada Agregada")
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Error al obtener máscaras de TabNet: {e}")
    print(f"Asegúrate de que tu capa de codificador TabNet se llame '{TABNET_ENCODER_NAME}'")